In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

# --- Interactive Jupyter Notebook for Whittaker-Shannon Sinc Interpolation ---

@widgets.interact(
    signal_type=widgets.Dropdown(
        options=[('Smooth Cosine Signal', 1), ('Complex Multi-frequency Signal', 2)],
        value=1,
        description='Signal Type:',
        style={'description_width': 'initial'}
    ),
    Fs=widgets.FloatSlider(value=200.0, min=100.0, max=500.0, step=25.0, description='Sampling $F_s$ (Hz):', style={'description_width': 'initial'}, layout=widgets.Layout(width='700px'))
)
def update_sinc_interpolation_plot(signal_type, Fs):
    print(f"Active Sampling Frequency: {Fs} Hz | Signal Mode: {signal_type}")
    
    Ts = 1.0 / Fs
    
    # Time axis setup
    t_start = -1.5 * Ts
    t_end = 5.5 * Ts
    t_cont = np.linspace(t_start, t_end, 2000)
    
    # Define underlying continuous signal based on selection
    if signal_type == 1:
        # Simple signal
        f_sig = 40.0
        x_cont = np.cos(2 * np.pi * f_sig * t_cont) + 0.5 * np.sin(2 * np.pi * (f_sig * 0.5) * t_cont)
    else:
        # Richer harmonic signal
        x_cont = 0.8 * np.cos(2 * np.pi * 30 * t_cont) + 0.6 * np.sin(2 * np.pi * 70 * t_cont)
        
    # Sample instances from n = -2 to 6
    n_values = np.arange(-2, 7)
    t_samples = n_values * Ts
    
    # Evaluate samples on the continuous signal
    if signal_type == 1:
        x_samples = np.cos(2 * np.pi * f_sig * t_samples) + 0.5 * np.sin(2 * np.pi * (f_sig * 0.5) * t_samples)
    else:
        x_samples = 0.8 * np.cos(2 * np.pi * 30 * t_samples) + 0.6 * np.sin(2 * np.pi * 70 * t_samples)

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.grid(True, linestyle=':', alpha=0.7)
    ax.set_xlim(t_start, t_end)
    ax.set_ylim(-1.4, 1.8)
    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(0, color='black', linewidth=1)
    
    ax.set_xlabel(r'Time $t$', fontsize=11)
    ax.set_ylabel(r'$x_r(t)$', fontsize=11)
    ax.set_title(r'Ideal Signal Reconstruction via Sinc Interpolation (Whittaker-Shannon Theorem)', fontsize=12, fontweight='bold')
    
    # Total reconstructed signal accumulator
    x_reconstructed = np.zeros_like(t_cont)
    
    # Plot individual scaled sinc pulses for each sample
    for n, t_n, x_n in zip(n_values, t_samples, x_samples):
        # Sinc function: sinc(x) = sin(pi*x)/(pi*x). numpy.sinc handles the normalized pi factor: sinc(x) = sin(pi*x)/(pi*x)
        # Therefore, to get standard unnormalized argument: np.sinc(Fs * (t_cont - t_n))
        sinc_pulse = x_n * np.np_sinc_equivalent_if_needed if False else x_n * np.sinc(Fs * (t_cont - t_n))
        x_reconstructed += sinc_pulse
        
        # Draw individual red sinc curves
        ax.plot(t_cont, sinc_pulse, color='red', alpha=0.6, linewidth=1.5)
        
        # Draw vertical stem lines from axis to sample point
        ax.plot([t_n, t_n], [0, x_n], color='black', linewidth=1, alpha=0.7)

    # Plot sample blue dots and ticks on axis
    for t_n, x_n in zip(t_samples, x_samples):
        ax.plot(t_n, 0, marker='o', markersize=5, color='#1f77b4', zorder=5) # zero-crossing reference
        ax.plot(t_n, x_n, marker='o', markersize=7, color='#1f77b4', zorder=6) # sample value

    # Plot final reconstructed signal as a dashed green curve
    ax.plot(t_cont, x_reconstructed, color='darkgreen', linestyle='--', linewidth=2.5, label='Reconstructed signal')
    
    # Custom x-axis ticks showing multiples of Ts
    custom_ticks = [n * Ts for n in range(-1, 6)]
    custom_labels = ['$-T_s$', '0', '$T_s$', '$2T_s$', '$3T_s$', '$4T_s$', '$5T_s$']
    ax.set_xticks(custom_ticks[:len(custom_labels)])
    ax.set_xticklabels(custom_labels[:len(custom_ticks)], fontsize=10)

    ax.legend(loc='upper right', fontsize=10)
    plt.show()

    # --- Educational Output Guide ---
    print("\n" + "="*95)
    print(" THEORETICAL INTERPOLATION GUIDE (WHITTAKER-SHANNON RECONSTRUCTION):")
    print("="*95)
    print(" - This notebook models the exact textbook diagram for ideal signal reconstruction in the time domain.")
    print(" - Each discrete sample x[n] (blue dots) acts as the scaling factor for a shifted sinc basis function (red curves).")
    print(" - Summing all overlapping sinc pulses perfectly rebuilds the continuous signal (green dashed curve).")
    print("="*95)

interactive(children=(Dropdown(description='Signal Type:', options=(('Smooth Cosine Signal', 1), ('Complex Mul…